In [10]:
import numpy as np
import pandas as pd
from flexible_emission_hmm import HMM, GaussianEmission, BaumWelchTrainer
from pathlib import Path

In [14]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "synthetic" / "samples.csv")
df = df.sort_values("time")

X = df[["data"]].to_numpy(dtype=float)

true_states = df["state"].to_numpy()

split = int(len(X) * 0.8)
X_train = X[:split]
X_test = X[split:]

model = HMM(
    emission=GaussianEmission(n_states=2, random_state=42),
    trainer=BaumWelchTrainer(max_iter=1000, tol=1e-2),
)

model.fit(X_train)

test_log_likelihood = model.score(X_test)
test_states = model.predict(X_test)
test_probabilities = model.predict_proba(X_test)

print("History:", model.history)
print("Converged:", model.trainer.converged)
print("Training iterations:", model.trainer.n_iter)
print("Test log likelihood per observation:",
      test_log_likelihood / len(X_test))

History: [-21584.750935299813, -19734.68339783936, -19600.975159327943, -19480.257852491042, -19389.592433582333, -19317.25123986241, -19250.497844464724, -19185.42959964813, -19123.031092930152, -19066.02790531178, -19017.264043770825, -18978.479245291117, -18949.682283838756, -18929.448978779645, -18915.753692930513, -18906.675100349927, -18900.709142215368, -18896.793266167944, -18894.215631312367, -18892.51090995797, -18891.377528808105, -18890.62007570402, -18890.111420039797, -18889.768370092697, -18889.53613835127, -18889.37841652379, -18889.271002023557, -18889.19767687096, -18889.14752310464, -18889.113161114194, -18889.089585615588, -18889.07339166639, -18889.062257178255, -18889.05459515167]
Converged: True
Training iterations: 33
Test log likelihood per observation: -2.3656566475237493


In [15]:
inferred_states = model.predict(X_test)
probabilities = model.predict_proba(X_test)

results = df.iloc[split:][["time", "state", "data"]].copy()
results["inferred_state"] = inferred_states
for k in range(model.n_states):
    results[f"prob_state_{k}"] = probabilities[:, k]

print("First 50 inferred regimes:", inferred_states[:50])
print(results.head(20).to_string(index=False))

First 50 inferred regimes: [0 1 1 1 1 0 1 0 0 1 1 1 0 1 1 1 0 1 1 1 0 0 1 0 0 0 1 0 0 1 1 0 1 1 1 1 1
 1 0 1 0 1 0 1 0 1 0 1 1 1]
 time  state      data  inferred_state  prob_state_0  prob_state_1
 8000      1  6.861333               0  1.000000e+00  2.436658e-64
 8001      1  2.456269               1  1.278541e-01  8.721459e-01
 8002      1  3.393289               1  3.565079e-03  9.964349e-01
 8003      1  5.221409               1  7.873797e-07  9.999992e-01
 8004      1  2.151497               1  2.518938e-01  7.481062e-01
 8005      0  1.222134               0  7.465064e-01  2.534936e-01
 8006      1  4.711155               1  2.064513e-05  9.999794e-01
 8007      0  0.634160               0  9.300525e-01  6.994747e-02
 8008      0 -0.618302               0  9.845621e-01  1.543791e-02
 8009      1  5.333671               1  6.042115e-07  9.999994e-01
 8010      1  2.542571               1  6.630530e-02  9.336947e-01
 8011      1  4.038219               1  3.599864e-04  9.996400e-01